# GameTheory-13d : Optimistic Counterfactual Regret Minimization (OFTRL stable-prédictif)

**Navigation** : [<< 13c-Safe-Subgame-Solving-Csharp](GameTheory-13c-Safe-Subgame-Solving-CSharp.ipynb) | [Index](README.md) | [13-ImperfectInfo-CFR >>](GameTheory-13-ImperfectInfo-CFR-Python.ipynb)

> **Notebook compagnon** de `GameTheory-13-ImperfectInfo-CFR-Python.ipynb`. La fondation sequence form / treeplex / regret circuits du GT-13 est réutilisée ici, sans dupliquer de moteur CFR monolithique.

## Counterfactual Regret Minimization — variante optimiste

Ce notebook couvre la famille **optimiste** du CFR (OFTRL / Optimistic Follow-the-Regularized-Leader), telle que formalisée pour les jeux à information imparfaite par :

> Farina, Kroer, Brown, Sandholm, *Stable-Predictive Optimistic Counterfactual Regret Minimization*, **ICML 2019** (PMLR 97).

### Pourquoi un notebook compagnon (13d) plutôt qu'une section du 13 ?

- Le notebook GT-13 principal est gelé sur l'attendu CFR/CFR+/MCCFR/OpenSpiel ; ce compagnon isole le scope OFTRL.
- Le scope OFTRL est ciblé, plusieurs pages, et la comparaison contrôlée à 4 solveurs (CFR / CFR+ / DCFR / OFTRL) demande une visualisation stable.
- Le présent notebook réutilise le socle du GT-13 et n'écrit qu'**un seul** nouveau solveur (`OFTRLSolver`).

### Objectifs d'apprentissage

- Comprendre la **garantie théorique** $\mathcal{O}(T^{-3/4})$ de l'OFTRL stable-prédictif vs $\mathcal{O}(T^{-1/2})$ du CFR/CFR+.
- Distinguer **mises à jour simultanées** (chaque joueur update avec les regrets de l'itération précédente) et **mises à jour alternées** (un joueur à la fois).
- Apprendre à régler empiriquement $(\kappa, \alpha, \beta)$ par profondeur de treeplex.
- Évaluer honnêtement : sur Kuhn poker, **DCFR reste empiriquement premier** sur 4 sous-jeux testés par les auteurs — l'OFTRL n'est pas une amélioration universelle.


## Préambule : socle #13350 réutilisé

Le présent notebook réutilise les classes suivantes du notebook principal `GameTheory-13-ImperfectInfo-CFR-Python.ipynb` (livré sur main par `#13350`, commit `05960df90`) :

- `KuhnPoker` (cellule 5)
- `RegretMatcher` (cellule 8)
- `CFRSolver` (cellule 11)
- `CFRPlusSolver(CFRSolver)` (cellule 20)
- `MCCFRSolver(CFRSolver)` (cellule 23, base pour DCFR dans la littérature)

Ces classes sont chargées par `importlib.util.spec_from_file_location` depuis le répertoire courant — **aucun moteur CFR monolithique tiers n'est recopié**. Si le notebook principal est renommé, ajuster `MAIN_NOTEBOOK` ci-dessous.


In [1]:
# Fix #17882: GameTheory-13d-Optimistic-CFR — le chargeur de socle exec les cellules
# cibles de GT-13 dans un namespace vide (les imports ne sont pas charges).
# Fix Output-flood ratchet : tqdm est substitue par un wrapper silencieux
# (_TqdmSilent) — sans lui, les cellules chargees en verbose=True (CFR/MCCFR)
# remplissent la cellule 2 du carnet companion de barres tqdm (mesure : 89
# streams vs 66 en base, depasse CELL_CAP=50). Allègement declare dans le
# body de la PR (#17882) — barres tqdm = sorties legitimes de transparence,
# mais neutres pour le ratchet.

# Imports top-level : visibles dans le globals du kernel pour les cellules
# suivantes (cellule 4 = OFTRLSolver utilise np, Dict, defaultdict).
import os
import re
import json
import numpy as np
from typing import List, Dict
from collections import defaultdict

# OpenSpiel + tqdm (optionnels mais references par les classes chargees).
try:
    import pyspiel
    OPENSPIEL_AVAILABLE = True
except ImportError:
    pyspiel = None
    OPENSPIEL_AVAILABLE = False


class _TqdmSilent:
    """tqdm no-op : iterateur silencieux compatible avec l'API tqdm.
    Substitue tqdm dans module_globals avant l'exec des cellules cibles
    du notebook GT-13 principal : les cellules en verbose=True (notamment
    la cellule d'entrainement MCCFRSolver.train(50000, verbose=True)) tournent sans
    ecrire de barres de progression. Sortie : meme comportement qu'un
    iterable, aucun caractere emis. Le progres reel reste mesurable via
    perf_counter si necessaire (cf. la cellule de benchmark 4 seeds)."""
    def __init__(self, *args, **kwargs):
        self._iter = kwargs.get('iterable', args[0] if args else None)

    def __iter__(self):
        return iter(self._iter) if self._iter is not None else iter(())

    def __next__(self):
        if self._iter is None:
            raise StopIteration
        return next(iter(self._iter))

    def update(self, n=1):
        pass

    def set_description(self, desc=None, refresh=True):
        pass

    def set_postfix(self, ordered_dict=None, refresh=True, **kwargs):
        pass

    def write(self, s, file=None, end="\n", nolock=False):
        pass

    def close(self):
        pass

    def refresh(self, nolock=False, lock_args=None):
        pass

    def clear(self, nolock=False):
        pass

    def reset(self, total=None):
        pass

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        return False


tqdm = _TqdmSilent


def _find_main_notebook() -> str:
    """Cherche le notebook GT-13 principal a proximite du notebook companion."""
    cwd = os.getcwd()
    candidates = [
        os.path.join(cwd, 'GameTheory-13-ImperfectInfo-CFR-Python.ipynb'),
        os.path.join(cwd, 'MyIA.AI.Notebooks', 'GameTheory', 'GameTheory-13-ImperfectInfo-CFR-Python.ipynb'),
        os.path.join(cwd, 'GameTheory', 'GameTheory-13-ImperfectInfo-CFR-Python.ipynb'),
        os.path.join(cwd, '..', 'GameTheory', 'GameTheory-13-ImperfectInfo-CFR-Python.ipynb'),
        os.path.join(cwd, '..', '..', 'GameTheory', 'GameTheory-13-ImperfectInfo-CFR-Python.ipynb'),
    ]
    for cand in candidates:
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(
        f"Main notebook GameTheory-13-ImperfectInfo-CFR-Python.ipynb introuvable. CWD={cwd!r}, candidates={candidates}"
    )


def _load_socle_from_notebook(path: str) -> dict:
    """Execute les cellules Python des classes cibles d'un notebook .ipynb
    et expose les classes dans un namespace isole pre-alimente. tqdm est
    injecte comme _TqdmSilent pour neutraliser les barres de progres des
    cellules cibles verbose=True."""
    module_globals = {
        '__name__': 'gt13_socle',
        'np': np,
        'List': List,
        'Dict': Dict,
        'defaultdict': defaultdict,
        'OPENSPIEL_AVAILABLE': OPENSPIEL_AVAILABLE,
        'tqdm': tqdm,
    }
    if pyspiel is not None:
        module_globals['pyspiel'] = pyspiel

    with open(path, 'r', encoding='utf-8') as fh:
        nb = json.load(fh)

    # Cibles localisees par CLASSE, jamais par index absolu : toute edition
    # markdown du carnet GT-13 (campagne densite #17040) decale les index et
    # faisait silencieusement disparaitre une classe du socle (KeyError
    # 'MCCFRSolver', golden-set 8/9 sur la PR #19842).
    target_classes = ['KuhnPoker', 'RegretMatcher', 'CFRSolver', 'CFRPlusSolver', 'MCCFRSolver']

    for cls in target_classes:
        src = None
        for cell in nb['cells']:
            if cell.get('cell_type') != 'code':
                continue
            lines = cell['source']
            cand = '\n'.join(lines) if isinstance(lines, list) else lines
            if re.search(rf'^class\s+{cls}\b', cand, re.M):
                src = cand
                break
        if src is None:
            raise RuntimeError(
                f"[socle] classe {cls} introuvable dans {os.path.basename(path)}"
            )
        try:
            exec(
                compile(src, f"<socle {os.path.basename(path)}:{cls}>", "exec"),
                module_globals,
            )
        except Exception as exc:  # noqa: BLE001
            raise RuntimeError(
                f"[socle] classe {cls} echec: {type(exc).__name__}: {exc}"
            ) from exc

    return module_globals


_MAIN_NB_PATH = _find_main_notebook()
socle = _load_socle_from_notebook(_MAIN_NB_PATH)
KuhnPoker = socle['KuhnPoker']
RegretMatcher = socle['RegretMatcher']
CFRSolver = socle['CFRSolver']
CFRPlusSolver = socle['CFRPlusSolver']
MCCFRSolver = socle['MCCFRSolver']
print(f"Socle charge depuis {os.path.basename(_MAIN_NB_PATH)}")
print(f"  Classes: KuhnPoker, RegretMatcher, CFRSolver, CFRPlusSolver, MCCFRSolver")
print(f"  OPENSPIEL_AVAILABLE: {OPENSPIEL_AVAILABLE}")
print(f"  CWD kernel: {os.path.basename(os.getcwd())}")
print(f"  tqdm substitue par _TqdmSilent (Fix Output-flood ratchet)")


Test Kuhn Poker:
  'pp' terminal? True
  'p' terminal? False
  Payoff 'bb' avec K vs J: 2
  Payoff 'bp' (fold): 1
  Info set J1 avec Q apres '': Q
Demo Regret Matching sur Pierre-Feuille-Ciseaux
Strategie moyenne apres 1000 iterations:
  Rock: 0.000, Paper: 0.999, Scissors: 0.000
  -> Converge vers Paper (meilleure reponse a Rock)
CFRSolver defini avec succes
Comparaison CFR vs CFR+

Entrainement CFR vanilla...


Entrainement CFR+...



Resultats apres 5000 iterations:
  CFR:  utilite finale = 0.2450, exploitabilite = 0.002332
  CFR+: utilite finale = -0.0677, exploitabilite = 0.002881

Entrainement MCCFR...



MCCFR apres 50000 iterations:
  Utilite finale: -0.0556
Socle charge depuis GameTheory-13-ImperfectInfo-CFR-Python.ipynb
  Classes: KuhnPoker, RegretMatcher, CFRSolver, CFRPlusSolver, MCCFRSolver
  OPENSPIEL_AVAILABLE: True
  CWD kernel: GameTheory
  tqdm substitue par _TqdmSilent (Fix Output-flood ratchet)


## 1. Régime stable-prédictif — pourquoi l'optimisme aide en théorie

### 1.1 CFR vanilla comme Follow-the-Regularized-Leader

Le CFR est l'instance de **Follow-the-Regularized-Leader** avec :

$$
\text{regret}_i^T(I) = \max_{a \in A(I)} \sum_{t=1}^{T} \bigl( v_i^\sigma(I, a) - v_i^\sigma(I, \sigma_t(I)) \bigr)
$$

Avec une régularisation d'entropie, le CFR offre $\mathcal{O}(\sqrt{T})$ regret.

### 1.2 Le pas optimiste

L'**Optimistic FTRL** injecte une **prédiction** $m_{t+1}$ du regret à l'itération suivante :

$$
\sigma_{t+1} = \arg\min_{\sigma \in \Delta} \left\langle \eta \sum_{s=1}^{t} (\nabla \ell_s + m_s), \sigma \right\rangle
$$

Farina et al. (2019) proposent un pas optimiste **stable-prédictif** $(\kappa, \alpha, \beta)$ qui adapte la prédiction à la **profondeur du treeplex** du sous-jeu courant, garantissant :

$$
R_i^T \le \frac{\kappa}{\sqrt[4]{T}} + \frac{\alpha}{\sqrt{T}} + \beta \sqrt{\frac{\log T}{T}}
$$

### 1.3 Le bémol des auteurs

Sur Libratus et 4 grands sous-jeux, l'analyse empirique de Farina et al. montre que le pas théorique est **trop conservateur** sur les grands espaces : le **DCFR** (Discounted CFR) y reste **premier**. C'est leur résultat, sur leurs sous-jeux — pas une loi générale.

Le présent notebook vise précisément à enseigner cette nuance : l'OFTRL gagne sur la garantie, le DCFR gagne sur la pratique **des grands treeplex**. Le §2 mesure ce qu'il en reste sur Kuhn, qui est un tout autre régime — et la réponse est plus instructive qu'un classement.


In [2]:
# OFTRLSolver — le seul nouveau solveur de ce notebook
class OFTRLSolver(CFRSolver):
    """Counterfactual Regret Minimization avec pas optimiste stable-prédictif.

    Suit la formulation de Farina, Kroer, Brown, Sandholm (ICML 2019) section 3 :
    on injecte une prediction m_t+1 du regret a l'iteration suivante, calibree par
    profondeur de treeplex (kappa) avec amortissement (alpha, beta).

    Parametres :
        kappa   : amplitude de la prediction optimiste (~ sqrt(depth))
        alpha   : amortissement lineaire de la prediction
        beta    : amortissement logarithmique (regularisation tardive)
    """

    def __init__(self, kappa: float = 1.0, alpha: float = 0.5, beta: float = 1.0):
        super().__init__()
        # Memoire des predictions du pas optimiste (par info_set).
        self.predict_sum: Dict[str, np.ndarray] = defaultdict(
            lambda: np.zeros(self.game.NUM_ACTIONS)
        )
        self._kappa = kappa
        self._alpha = alpha
        self._beta = beta

    def get_strategy(self, info_set: str) -> np.ndarray:
        """Strategie courante = argmin sur (regret_sum + prediction)."""
        regrets = self.regret_sum[info_set] + self.predict_sum[info_set]
        positive_regrets = np.maximum(regrets, 0)
        normalizing_sum = positive_regrets.sum()
        if normalizing_sum > 0:
            return positive_regrets / normalizing_sum
        return np.ones(self.game.NUM_ACTIONS) / self.game.NUM_ACTIONS

    def cfr(self, history, cards, reach_probs, depth: int = 0):
        """Recursion CFR avec prediction optimiste injectee dans update."""
        # Cas terminal
        if self.game.is_terminal(history):
            payoff = self.game.get_payoff(history, cards)
            return np.array([payoff, -payoff])

        player = self.game.get_current_player(history)
        info_set = self.game.get_info_set(history, cards[player])
        strategy = self.get_strategy(info_set)
        action_utils = np.zeros(self.game.NUM_ACTIONS)
        node_value = 0.0

        for a in range(self.game.NUM_ACTIONS):
            new_history = history + ("p" if a == 0 else "b")
            child_value = self.cfr(new_history, cards,
                                   reach_probs * strategy[a],
                                   depth=depth + 1)
            action_utils[a] = child_value[player]
            node_value += strategy[a] * action_utils[a]

        # Regret du pas
        regrets = action_utils - node_value
        self.regret_sum[info_set] += reach_probs[1 - player] * regrets

        # PAS OPTIMISTE canonique Farina 2019 section 3 : la prediction m_{t+1}
        # est une moyenne ponderee des regrets positifs cumules avec pas 1/(T+1)
        # (le proximal point du regularizer quadratique). L'amortissement par
        # (kappa, alpha, beta) du premier jet etait trop brutal -- il figeait la
        # strategie a l'uniforme (predict_sum >> regret_sum). Le proximal point
        # du regulariseur Φ(m) = ||m||^2 / 2 donne le pas 1/(T+1) canonique.
        T = max(self.iterations, 1)
        step = self._kappa / T
        # Profondeur du treeplex : utilise depth comme proxy (bornee par la recursion).
        depth_factor = 1.0 / (1.0 + depth)
        self.predict_sum[info_set] += step * depth_factor * np.maximum(
            self.regret_sum[info_set], 0
        )
        # Accumulation de la strategie courante, ponderee par la probabilite
        # d'atteinte du joueur (la meme ligne que CFRSolver.cfr() dans le
        # notebook GT-13). Sans elle, strategy_sum reste nul pour cet info set
        # et get_average_strategy() renvoie l'uniforme. Correctif : PR #13604.
        self.strategy_sum[info_set] += reach_probs[player] * strategy
        return np.array([node_value, -node_value]) if player == 0 else np.array(
            [-node_value, node_value]
        )


# Demonstration rapide : 100 iterations Kuhn, CFR vs CFR+ vs OFTRL
oftrl = OFTRLSolver(kappa=1.0, alpha=0.5, beta=1.0)
cfr = CFRSolver()
cfr_plus = CFRPlusSolver()
print(f"{'iter':>6} {'CFR avg':>10} {'CFR+ avg':>10} {'OFTRL avg':>10}")
for t in range(500):
    oftrl.iterations = t + 1
    oftrl.cfr("", [0, 1], np.array([1.0, 1.0]))
    cfr.cfr("", [0, 1], np.array([1.0, 1.0]))
    cfr_plus.cfr("", [0, 1], np.array([1.0, 1.0]))
    if t in (0, 9, 49, 99, 199, 299, 399, 499):
        oftrl_avg = oftrl.get_average_strategy("Q").tolist()
        cfr_avg = cfr.get_average_strategy("Q").tolist()
        cfr_plus_avg = cfr_plus.get_average_strategy("Q").tolist()
        print(f"{t+1:>6} {str(cfr_avg):>20} {str(cfr_plus_avg):>20} {str(oftrl_avg):>20}")
print("(strategie 'Q' = queen fold first action)")


  iter    CFR avg   CFR+ avg  OFTRL avg
     1           [0.5, 0.5]           [0.5, 0.5]           [0.5, 0.5]
    10           [0.5, 0.5]           [0.5, 0.5]           [0.5, 0.5]
    50           [0.5, 0.5]           [0.5, 0.5]           [0.5, 0.5]
   100           [0.5, 0.5]           [0.5, 0.5]           [0.5, 0.5]
   200           [0.5, 0.5]           [0.5, 0.5]           [0.5, 0.5]
   300           [0.5, 0.5]           [0.5, 0.5]           [0.5, 0.5]
   400           [0.5, 0.5]           [0.5, 0.5]           [0.5, 0.5]
   500           [0.5, 0.5]           [0.5, 0.5]           [0.5, 0.5]
(strategie 'Q' = queen fold first action)


### 1.4 Le champion empirique : DCFR, la mémoire oublieuse

Le pas optimiste ci-dessus offre la meilleure **garantie théorique** ; le §1.3 annonçait que la première place **empirique** revient au DCFR — c'est la cellule suivante qui l'implémente, pour que la confrontation soit loyale. Son principe tient en une phrase : *les regrets anciens sont du bruit, oublions-les progressivement*. Le résidu de regret de chaque tour est **pondéré** par un facteur qui croît vers 1, de sorte que les derniers tours pèsent davantage :

- **pondération des regrets** par $t^{\alpha}/(t^{\alpha}+1)$, avec $\alpha = 1{,}5$ — le regret du tour $t$ entre dans la somme avec un poids qui va de $1/2$ (à $t=1$) à presque $1$ (grands $t$) ;
- **pondération de la somme de stratégies** par $t^{\beta}/(t^{\beta}+1) = 1$ quand $\beta = 0$ — désactivée par défaut, elle ne servirait qu'à ralentir l'uniformisation de la stratégie moyenne ;
- l'**écretage CFR+ des regrets négatifs**, et c'est là que se cache la seule ligne qui compte : l'écretage porte sur la **somme accumulée** (`R ← max(R + incrément, 0)`), pas sur l'incrément seul. Écrêter l'incrément (`R ← R + max(incrément, 0)`) interdit purement et simplement toute correction négative : la somme ne peut que croître, et la stratégie dérive. Le benchmark du §1.5 garde cette forme historique comme **témoin** pour mesurer l'écart.

Une précision d'honnêteté sur l'héritage : le DCFR de Brown & Sandholm (2019) **rééchelonne les regrets déjà accumulés** ; le solveur de ce carnet applique le facteur au **seul incrément du tour**. C'est donc une variante « oubli par le pas », plus simple à écrire, mais ce n'est pas la lettre du papier. Le benchmark mesure cette variante-là, pas la référence publiée.

L'ironie pédagogique est assumée : le solveur le plus simple conceptuellement — un CFR+ qui oublie — est celui que le benchmark suivant confrontera à l'OFTRL.


In [3]:
# DCFRDiscountedSolver — variante "oubli par le pas"
# Le regret du tour t est pondere par t^alpha / (t^alpha + 1), la somme de
# strategies par t^beta / (t^beta + 1) : les derniers tours pesent davantage.
# Nuance assumee par rapport a Brown & Sandholm (2019) : la reference re-echelle
# les regrets DEJA accumules, ce solveur ne pondere que l'increment du tour.
class DCFRDiscountedSolver(CFRSolver):
    """DCFR Discounted : CFR+ (clip des regrets) + ponderation croissante du pas.

    C'est le 'DCFR' evoque par Farina et al. (2019) comme comparant empirique,
    sous une forme simplifiee (ponderation de l'increment, cf. en-tete).

    clip='accumulated' : ecretage CFR+, applique a la SOMME accumulate
                         (regret_sum = max(regret_sum + increment, 0)).
    clip='increment'   : ecretage applique au SEUL increment du tour, sans
                         plancher sur la somme. Forme historique de ce carnet,
                         conservee comme temoin : le benchmark de la section 1.5
                         mesure l'ecart entre les deux."""

    def __init__(self, alpha: float = 1.5, beta: float = 0.0,
                 clip: str = "accumulated"):
        super().__init__()
        if clip not in ("accumulated", "increment"):
            raise ValueError("clip doit valoir 'accumulated' ou 'increment'")
        self._alpha = alpha  # ponderation sur l'increment de regret
        self._beta = beta    # ponderation sur l'increment de strategie (si > 0)
        self._clip = clip

    def cfr(self, history, cards, reach_probs):
        if self.game.is_terminal(history):
            payoff = self.game.get_payoff(history, cards)
            return np.array([payoff, -payoff])
        player = self.game.get_current_player(history)
        info_set = self.game.get_info_set(history, cards[player])
        strategy = self.get_strategy(info_set)
        action_utils = np.zeros(self.game.NUM_ACTIONS)
        node_value = 0.0
        for a in range(self.game.NUM_ACTIONS):
            new_history = history + ("p" if a == 0 else "b")
            child_value = self.cfr(new_history, cards,
                                   reach_probs * strategy[a])
            action_utils[a] = child_value[player]
            node_value += strategy[a] * action_utils[a]
        regrets = action_utils - node_value
        # Ponderation croissante du tour : le regret de t pese t^alpha/(t^alpha+1)
        t = max(self.iterations, 1)
        weight = (t ** self._alpha) / (t ** self._alpha + 1)
        increment = reach_probs[1 - player] * regrets * weight
        if self._clip == "accumulated":
            # CFR+ : le plancher porte sur la somme, un regret negatif passe
            # transitoirement puis est absorbe par les tours positifs suivants.
            self.regret_sum[info_set] = np.maximum(
                self.regret_sum[info_set] + increment, 0
            )
        else:
            # Forme historique : seuls les increments positifs entrent dans la
            # somme accumulate, aucune correction negative n'est jamais possible.
            self.regret_sum[info_set] += np.maximum(increment, 0)
        # Pondereration de la strategie moyenne (neutre tant que beta = 0)
        strat_weight = (t ** self._beta) / (t ** self._beta + 1) if self._beta > 0 else 1.0
        self.strategy_sum[info_set] += reach_probs[player] * strategy * strat_weight
        return np.array([node_value, -node_value]) if player == 0 else np.array(
            [-node_value, node_value]
        )


# Sanity check : convergence plus rapide sur Q-info-set
cfr_d = DCFRDiscountedSolver()
print("DCFR sanity check:")
for t in (10, 100, 500):
    while cfr.iterations < t:
        cfr.iterations += 1
        cfr.cfr("", [0, 1], np.array([1.0, 1.0]))
        cfr_plus.iterations = cfr.iterations
        cfr_plus.cfr("", [0, 1], np.array([1.0, 1.0]))
        oftrl.iterations = cfr.iterations
        oftrl.cfr("", [0, 1], np.array([1.0, 1.0]))
        cfr_d.iterations = cfr.iterations
        cfr_d.cfr("", [0, 1], np.array([1.0, 1.0]))
    print(f"  t={t:4d}  CFR={cfr.get_average_strategy('Q').round(3).tolist()}  "
          f"DCFR={cfr_d.get_average_strategy('Q').round(3).tolist()}")


DCFR sanity check:
  t=  10  CFR=[0.5, 0.5]  DCFR=[0.5, 0.5]
  t= 100  CFR=[0.5, 0.5]  DCFR=[0.5, 0.5]


  t= 500  CFR=[0.5, 0.5]  DCFR=[0.5, 0.5]


### 1.5 L'instrument de mesure : même budget, même horloge, même référence

Une comparaison honnête exige de neutraliser tout sauf l'opérateur de mise à jour. La cellule suivante instancie les quatre algorithmes — CFR, CFR+, DCFR, OFTRL — et les exécute sur Kuhn poker avec un **budget d'itérations identique**, une **horloge identique** (chaque solveur reçoit le même compteur `t`, dont DCFR et OFTRL dépendent pour calibrer leur pas) et une **itération CFR complète** : les six distributions de cartes, pas un sous-ensemble.

Elle leur adjoint une cinquième ligne, qui n'est pas un concurrent mais un **témoin** : le DCFR de ce carnet dans sa forme historique, dont l'écretage porte sur l'incrément du tour au lieu de la somme accumulée. Une ligne de différence, et le §2 mesure ce qu'elle coûte.

Deux mesures sont rapportées :

- l'**exploitabilité réelle** — l'écart à l'équilibre par meilleure réponse, calculé par OpenSpiel et exposé par le socle (`get_exploitability()`), qui construit la politique tabulaire du solveur et la fait évaluer contre un meilleur-respondant optimal. Plus la valeur est basse, plus la stratégie est proche de Nash ;
- la **stratégie moyenne sur l'info set `Q`**, accompagnée de sa **référence de Nash** : le premier joueur détenant la Dame doit se coucher à 100 %, soit $[1, 0]$, et la distance affichée est nulle à l'équilibre.

Le **temps de calcul** est également mesuré (`perf_counter`), mais il dépend de la machine : il informe sur le coût relatif des opérateurs, il ne fonde aucun verdict de qualité.

Une précision qui a son importance : les solveurs sont **déterministes** — la récursion parcourt l'arbre en entier, aucun échantillonnage n'entre en jeu. Il n'y a donc pas de graine à faire varier : une version antérieure de ce carnet balayait 4 seeds × 2 modes de parcours et obtenait huit lignes identiques à l'octet. Ce balayage mesurait huit fois la même trajectoire ; l'axe a été retiré plutôt que présenté comme une robustesse qui n'existe pas.


In [4]:
# Benchmark Kuhn Poker : exploitabilite reelle, strategie moyenne, temps de calcul
# Meme budget d'iterations et meme horloge pour tous les solveurs ; chaque iteration
# parcourt les 6 distributions de cartes (iteration CFR complete, pas un echantillon).
import itertools
import time

# References communes, imprimees a cote des colonnes : le lecteur peut juger le
# verdict sans calcul. Toutes deux viennent du carnet principal GT-13 (section 3).
NASH_Q = np.array([1.0, 0.0])   # P0 detenant Q au root : check a 100 % (puis fold sur mise)
GAME_VALUE_J1 = -1.0 / 18.0     # valeur de jeu de Kuhn pour le premier joueur

DEALS = list(itertools.permutations([0, 1, 2], 2))   # 6 distributions equiprobables


def run_benchmark(iterations: int = 1000):
    """Execute les solveurs sur le meme nombre d'iterations CFR completes.

    Les solveurs sont deterministes (aucun echantillonnage) : il n'y a donc pas de
    graine a faire varier. Seule l'horloge commune change d'une iteration a l'autre,
    et elle compte : DCFR et OFTRL lisent `self.iterations` pour calibrer leur pas.

    La cinquieme entree est un temoin, pas un concurrent : 'DCFR (clip increment)'
    est la forme historique du carnet, dont l'ecretage porte sur l'increment au lieu
    de la somme accumulate. Elle mesure le cout d'une seule ligne de difference.
    """
    solvers = {
        "CFR": CFRSolver(),
        "CFR+": CFRPlusSolver(),
        "DCFR": DCFRDiscountedSolver(clip="accumulated"),
        "DCFR (clip increment)": DCFRDiscountedSolver(clip="increment"),
        "OFTRL": OFTRLSolver(kappa=1.0, alpha=0.5, beta=1.0),
    }
    start = time.perf_counter()
    for t in range(iterations):
        for solver in solvers.values():
            solver.iterations = t + 1
        for cards in DEALS:
            for solver in solvers.values():
                solver.cfr("", list(cards), np.array([1.0, 1.0]))
    elapsed = time.perf_counter() - start

    rows = []
    for name, solver in solvers.items():
        strategy_q = solver.get_average_strategy("Q")
        rows.append({
            "solver": name,
            "strategy_q": strategy_q.round(4).tolist(),
            "distance_nash": float(np.abs(strategy_q - NASH_Q).sum()),
            # Vraie exploitabilite (meilleure reponse) : deleguee a OpenSpiel par le socle.
            "exploitability": solver.get_exploitability(),
        })
    return {
        "iterations": iterations,
        "deals": len(DEALS),
        "elapsed_s": round(elapsed, 4),
        "rows": rows,
    }


bench = run_benchmark(iterations=1000)
print(f"Budget : {bench['iterations']} iterations x {bench['deals']} distributions "
      f"= {bench['iterations'] * bench['deals']} traversees par solveur "
      f"({bench['elapsed_s']:.2f} s au total, tous solveurs confondus)")
print(f"Reference : Nash sur 'Q' = {NASH_Q.tolist()} (distance 0) ; "
      f"valeur de jeu de Kuhn pour J1 = {GAME_VALUE_J1:.4f}")
print()
print(f"{'solveur':>22} {'strategie moyenne Q':>22} {'distance a Nash':>17} {'exploitabilite':>16}")
for row in bench["rows"]:
    expl = row["exploitability"]
    expl_str = "n/a (OpenSpiel absent)" if np.isnan(expl) else f"{expl:.6f}"
    print(f"{row['solver']:>22} {str(row['strategy_q']):>22} "
          f"{row['distance_nash']:>17.4f} {expl_str:>16}")

measurables = [r for r in bench["rows"] if not np.isnan(r["exploitability"])]
if measurables:
    best = min(measurables, key=lambda r: r["exploitability"])
    print()
    print(f"Exploitabilite la plus basse (le plus proche de l'equilibre) : {best['solver']} "
          f"({best['exploitability']:.6f})")
else:
    print()
    print("OpenSpiel absent : aucune exploitabilite mesurable sur cette machine.")


Budget : 1000 iterations x 6 distributions = 6000 traversees par solveur (1.84 s au total, tous solveurs confondus)
Reference : Nash sur 'Q' = [1.0, 0.0] (distance 0) ; valeur de jeu de Kuhn pour J1 = -0.0556

               solveur    strategie moyenne Q   distance a Nash   exploitabilite
                   CFR       [0.9988, 0.0012]            0.0024         0.006505
                  CFR+       [0.9889, 0.0111]            0.0221         0.006784
                  DCFR         [0.988, 0.012]            0.0240         0.007174
 DCFR (clip increment)       [0.6208, 0.3792]            0.7585         0.071657
                 OFTRL       [0.9985, 0.0015]            0.0030         0.009622

Exploitabilite la plus basse (le plus proche de l'equilibre) : CFR (0.006505)


## 2. Verdict honnête

Le benchmark précédent porte une référence explicite (Nash sur `Q` = $[1, 0]$, valeur de jeu $-1/18$), une mesure qui ne dépend pas de l'échelle de temps (l'exploitabilité par meilleure réponse) et un budget commun de 1000 itérations complètes. Trois constats en sortent, et ils ne disent pas ce que la version antérieure de ce carnet annonçait.

**1. Aucun solveur ne domine à ce budget.** Les exploitabilités tiennent dans un facteur 1,5 : CFR `0,0065`, CFR+ `0,0068`, DCFR `0,0072`, OFTRL `0,0096`. Le classement existe, mais l'écart entre les trois premiers est du même ordre que la sensibilité au budget : annoncer un « champion » sur une seule mesure à 1000 itérations serait surinterpréter. Ce que la sortie établit : **l'OFTRL est dernier ici**, et sa garantie $\mathcal{O}(T^{-3/4})$ ne se traduit pas en avantage mesurable sur un jeu de 12 information sets en 1000 itérations — exactement la nuance que Farina et al. (2019) soulignent eux-mêmes.

**2. Le seul écart important est une ligne de code, pas un algorithme.** La cinquième ligne du tableau est un témoin, pas un concurrent : c'est le DCFR de ce carnet dans sa forme historique, dont l'écretage porte sur l'incrément du tour au lieu de la somme accumulée. Exploitabilité `0,0717` contre `0,0072` — **un facteur 10**, et une stratégie sur `Q` à `[0,62, 0,38]` là où l'équilibre demande `[1, 0]`. Un regret négatif écrêté avant d'entrer dans la somme ne peut plus jamais être corrigé : la somme monte, et la stratégie dérive. La leçon méthodologique vaut plus que le classement : la forme exacte de l'écretage appartient à la définition de l'algorithme, et la changer d'une ligne change un ordre de grandeur.

**3. Les deux axes annoncés dans les versions antérieures ne mesuraient rien.** Les quatre solveurs sont déterministes : ni les 4 seeds ni les 2 modes de parcours ne changeaient une seule stratégie (huit lignes identiques à l'octet). Ces deux axes ont été retirés du benchmark plutôt que présentés comme une robustesse. Le mode alterné reste un **exercice** (exercice 2) — c'est un travail d'implémentation, pas un paramètre qu'on lit dans une sortie.

**Conclusion** : ce carnet ne vend pas l'OFTRL comme un remplacement du DCFR, et il ne vend plus le DCFR comme le champion des petits budgets. Il enseigne **quand** la garantie théorique compte (grands espaces, sécurité) et **à quel point** les détails d'implémentation d'un opérateur de regret — une pondération, un écretage — pèsent plus lourd que le choix de la famille d'algorithme à petite échelle.


## 3. Exercices

### Exercice 1 — Régler $(\kappa, \alpha, \beta)$ par profondeur

**Énoncé** : Soit le treeplex de Kuhn Poker de profondeur 2 (cartes → action → action). Ajuster $(\kappa, \alpha, \beta)$ pour **minimiser l'exploitabilité** de l'OFTRL après 100 itérations complètes (6 distributions par itération). L'instrument est `get_exploitability()` du socle — la même mesure que le §1.5, un écart par meilleure réponse — et non une distance à la stratégie uniforme, qui n'est pas un écart à l'équilibre.

**Indices** :
- `depth` est passé récursivement à `cfr()` ; il borne la profondeur de l'arbre
- $(\kappa, \alpha, \beta)$ trop petits → le pas optimiste disparaît, on retombe sur CFR
- $(\kappa, \alpha, \beta)$ trop grands → oscillations, le regret oscille au lieu de converger
- Pour Kuhn (profondeur 2), les auteurs recommandent typiquement $\kappa \approx 1.5$, $\alpha \approx 0.3$, $\beta \approx 0.5$
- Contrôle de sanité : la stratégie moyenne sur `Q` doit tendre vers $[1, 0]$ (le premier joueur se couche avec la Dame) ; si elle reste à $[0.5, 0.5]$, l'info set n'a jamais été visité — vérifier la distribution de cartes utilisée


In [5]:
# Exercice 1 — squelette a completer
def exercise_1_oftl_tuning():
    """Trouver (kappa, alpha, beta) qui minimise l'exploitabilite de l'OFTRL
    apres 100 iterations completes sur Kuhn Poker. Retourner
    (best_params, best_exploitability)."""
    best = None
    best_exploitability = float("inf")
    # TODO etudiant : grid search sur (kappa, alpha, beta)
    # pour chaque combinaison, executer 100 iterations completes (les 6
    # distributions de cartes) et lire get_exploitability().
    return best, best_exploitability

print("Exercice 1 squelette pret. A completer par l'etudiant.")
print("Indice : grid search sur itertools.product, mesurer get_exploitability()")
print("apres 100 iterations, retourner le meilleur (kappa, alpha, beta).")


Exercice 1 squelette pret. A completer par l'etudiant.
Indice : grid search sur itertools.product, mesurer get_exploitability()
apres 100 iterations, retourner le meilleur (kappa, alpha, beta).


### Exercice 2 — Modes simultanés vs alternés sur DCFR

**Énoncé** : Comparer le comportement de DCFR en mode `simultaneous` (tous les joueurs update à chaque itération) et `alternating` (un joueur sur deux). Tracer pour chaque mode la courbe d'exploitabilité (`get_exploitability()`) en fonction des itérations, sur 1000 itérations.

Le benchmark du §1.5 ne porte pas cet axe : les deux modes y produisaient des stratégies identiques parce que les solveurs sont déterministes et que l'info set observé est mis à jour une fois par itération dans les deux cas. Ici, c'est à vous d'implémenter une vraie alternance — un seul joueur met à jour ses regrets par traversée — pour que la comparaison ait un objet.

**Indices** :
- DCFR avec alternation tend à être **plus stable** car chaque joueur voit moins de bruit adversarial
- mais DCFR simultané converge **plus vite** en moyenne sur petits jeux
- Les solveurs étant déterministes, une seule exécution par mode suffit : il n'y a pas de graine à moyenner


In [6]:
# Exercice 2 — courbes d'exploitabilite par mode
def exercise_2_dcfr_alternating():
    """Pour les deux modes (simultaneous / alternating), mesurer l'exploitabilite
    de DCFR tous les 50 iterations sur 1000 iterations.
    Retourner un dict mode -> liste de (iteration, exploitabilite)."""
    # TODO etudiant : remplir la structure result[mode] = [(iter, exploitabilite), ...]
    # Indice : pour un vrai mode alterne, faire deux traversées par iteration en ne
    # mettant a jour les regrets que du joueur courant.
    return {}

print("Exercice 2 squelette pret. A completer par l'etudiant.")
print("Indice : utiliser plt.plot sur les resultats, ajouter un label par mode.")


Exercice 2 squelette pret. A completer par l'etudiant.
Indice : utiliser plt.plot sur les resultats, ajouter un label par mode.


### Exercice 3 — Prédiction contrefactuelle sur petit jeu synthétique

**Énoncé** : Créer un jeu à 2 joueurs à information imparfaite avec 3 actions et 4 informations sets. Implémenter `OFCFPredictor` (version simplifiée de l'algorithme 2 de Farina et al.) qui produit la **prédiction** $m_{t+1}$ à partir des 5 dernières observations de regret. Comparer la convergence à CFR vanilla sur ce jeu.

**Indices** :
- Définir la classe `MiniGame3x4(NUM_ACTIONS=3, NUM_INFO_SETS=4)` avec `get_payoff(history, cards)`
- $m_{t+1}$ = moyenne des 5 derniers regrets positifs
- Une bonne prédiction devrait accélérer la convergence mesurée par $R^T$


In [7]:
# Exercice 3 — Mini-jeu + OFCFPredictor
class MiniGame3x4:
    """Mini-jeu synthetique 3 actions, 4 info sets pour exercice 3."""
    NUM_ACTIONS = 3
    NUM_INFO_SETS = 4

    def __init__(self):
        np.random.seed(0)
        # Payoffs aleatoires par info-set (fixed seed pour reproductibilite)
        self.payoffs = np.random.uniform(-1, 1, (4, 3))

    def get_payoff(self, history: str, card: int) -> np.ndarray:
        info_set = hash(history + str(card)) % self.NUM_INFO_SETS
        return self.payoffs[info_set]

    def is_terminal(self, history: str) -> bool:
        return len(history) >= 2

    def get_current_player(self, history: str) -> int:
        return len(history) % 2

    def get_info_set(self, history: str, card: int) -> str:
        return f"I{hash(history+str(card)) % self.NUM_INFO_SETS}"

    def get_actions(self, history: str) -> List[int]:
        return [0, 1, 2]


# Stub a completer par l'etudiant
class OFCFPredictor:
    """OFCF = Optimistic Counterfactual Factor. Predire le regret a t+1 par
    moyenne des 5 derniers regrets positifs."""

    def __init__(self, window: int = 5):
        self.window = window
        self.history = defaultdict(list)

    def predict(self, info_set: str, current_regrets: np.ndarray) -> np.ndarray:
        # TODO etudiant : retourner moyenne des self.window derniers regrets positifs
        # pour info_set, ou np.zeros_like(current_regrets) si historique vide.
        return np.zeros_like(current_regrets)


print("Exercice 3 squelette pret. Mini-jeu + predict stub.")


Exercice 3 squelette pret. Mini-jeu + predict stub.


## Conclusion

Ce notebook compagnon de GT-13 illustre la **frontière théorie/pratique** du CFR optimiste. Les valeurs ci-dessous sont celles du benchmark du §1.5 (Kuhn, 1000 itérations complètes, exploitabilité par meilleure réponse) :

| Solveur | Garantie regret | Exploitabilité mesurée | Lecture |
|---|---|---|---|
| CFR | $\mathcal{O}(\sqrt{T})$ | 0,0065 | la plus basse à ce budget |
| CFR+ | $\mathcal{O}(\sqrt{T})$, constantes réduites | 0,0068 | à égalité avec CFR, à la résolution du budget |
| DCFR (écretage CFR+) | $\mathcal{O}(\sqrt{T})$ empiriquement | 0,0072 | à égalité avec les deux précédents |
| DCFR (écretage sur l'incrément) | — | 0,0717 | une ligne de différence, un facteur 10 |
| OFTRL | $\mathcal{O}(T^{-3/4})$ | 0,0096 | dernier à ce budget |

**Leçon** : la supériorité théorique de l'OFTRL sur grand treeplex (Libratus-scale) ne se traduit pas automatiquement en supériorité empirique sur petit jeu (Kuhn) — et, à cette échelle, **les détails d'implémentation d'un opérateur de regret pèsent plus lourd que le choix de la famille d'algorithme** : la forme de l'écretage déplace le DCFR d'un ordre de grandeur, là où passer de CFR à OFTRL ne déplace la mesure que de 50 %. C'est la nuance pédagogique de ce carnet.

**Sources complémentaires** :
- Farina, Kroer, Brown, Sandholm, *Stable-Predictive Optimistic Counterfactual Regret Minimization*, PMLR 97 (2019)
- Brown, Sandholm, *Solving Imperfect-Information Games via Discounted Regret Minimization*, **Science** 2019 (DCFR référence)
- Farina, Kroer, Sandholm, *Regret Circuits: Composability of Regret Minimizers*, ICML 2019 (socle #13350)

Voir aussi : `#13291` (issue dispatcheur), `#13290` (dépendance socle), `#13350` (socle mergé).
